# Making SIMSOPT GPU native: device L-BFGS replay sweep

Select **Runtime > Change runtime type > GPU**, then run all cells. This experiment reloads the 207-variable, independently qualified augmented-Lagrangian warm start from the committed A100 analysis. It does not rerun the augmented Lagrangian. Eight fully device-resident L-BFGS candidates compare history sizes 10, 20, 50, and 100 with target-checkpoint patience 15 and 25. Every candidate is measured with the SIMSOPT CPU oracle; the winning surface and coils are exported for ParaView.

In [ ]:
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("No NVIDIA GPU is attached. Select Runtime > Change runtime type > T4 GPU, disconnect the old runtime, and reconnect.")
subprocess.run([nvidia_smi], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert jax.config.jax_enable_x64, report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu/test_lbfgs.py", "tests/gpu/test_device_lbfgs_replay_benchmark.py", "tests/gpu/test_local_residual_augmented_lagrangian_benchmark.py"], cwd=repo, check=True)

In [ ]:
artifact_root = Path("/content/simsopt-device-lbfgs-replay")
artifact_root.mkdir(exist_ok=True)
result_path = artifact_root / "device-lbfgs-replay-sweep.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/benchmark_device_lbfgs_replay.py", "--history-sizes", "10", "20", "50", "100", "--checkpoint-patiences", "15", "25", "--infeasible-patience", "15", "--minimum-relative-improvement", "1e-3", "--max-iterations", "150", "--max-line-search-iterations", "30", "--target-relative-tolerance", "0.10", "--quadratic-flux-target", "1e-5", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output", str(result_path)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_path.read_text())
assert result["schema_version"] == 1
assert result["workflow"] == "device_lbfgs_replay_sweep"
assert result["execution_platform"] == "gpu"
assert result["sweep"]["candidate_count"] == 8
assert result["sweep"]["history_sizes"] == [10, 20, 50, 100]
assert result["sweep"]["checkpoint_patiences"] == [15, 25]
for candidate in result["candidates"]:
    assert candidate["optimization"]["device_resident"]
    assert candidate["optimization"]["host_callbacks"] == 0
    assert {"objective", "quadratic_flux", "normalized_normal_field", "coil_constraints"} <= candidate["final_metrics"].keys()
    selection = candidate["optimization"]["checkpoint_selection"]
    assert selection["candidate_count"] >= 1
if result["winner"] is not None:
    assert result["winner"]["scientifically_validated"]
    for artifact in result["visualization"].values():
        if not isinstance(artifact, str) or not artifact.endswith((".vts", ".vtu")):
            continue
        path = artifact_root / artifact
        assert path.is_file() and path.stat().st_size > 0, path
print(json.dumps({"qualified_indices": result["qualified_indices"], "winner_index": result["winner_index"], "scientifically_validated": result["scientifically_validated"], "all_gates_passed": result["all_gates_passed"]}, indent=2))

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-device-lbfgs-replay", "zip", artifact_root)
files.download(archive)

## What to send back

Send the downloaded `simsopt-device-lbfgs-replay.zip`. The archive is retained even if no candidate passes every gate.